In [1]:
sample_string = "This is the first step of Kirti Pogra to crack 8 LPA"

In [4]:
raw_bytes = sample_string.encode('utf-8')
tokens = list(raw_bytes)

In [52]:
sample_string[raw_bytes.find(115)]

's'

In [8]:
len(tokens), len(sample_string)

(52, 52)

In [12]:
tokens[0]

84

In [15]:
def get_stats(tokens):
    total_len = len(tokens) -1
    adj_pairs = dict()
    for i in range(total_len):
        k = (tokens[i],tokens[i+1])
        v = adj_pairs.get(k,0) +1
        adj_pairs[k] =v

    return adj_pairs


In [17]:
stats = get_stats(tokens)

In [19]:
# get the max count token pair

best_pair = max(stats,key=stats.get)
best_pair

(105, 115)

In [36]:
def merge(ids, best_pair, new_ids,new_idx):
    total_len = len(ids)
    i=0
    while i< total_len:
        if i<len(ids)-1 and (ids[i],ids[i+1]) == best_pair:
            new_ids.append(new_idx)  #255+1
            i+=2
        else:
            new_ids.append(ids[i])
            i+=1

    return new_ids

In [ ]:
merge_token = merge(tokens,best_pair,[],256)

# now here we have a first even rule for the tokenizer which is (105, 115) = 256

best_pair

(105, 115)

In [38]:
tokens[:10],best_pair

([84, 104, 105, 115, 32, 105, 115, 32, 116, 104], (105, 115))

In [39]:
len(merge_token),len(tokens)

(50, 52)

In [59]:
merge_rules = {}
tokens = list(raw_bytes)

num_merges = 10


In [60]:
vocabulary = {idx:bytes([idx]) for idx in range(256)}


In [61]:
for i in range(num_merges):
    print("Before new rule:", len(tokens))
    # calculate adjacent pairs 
    stats = get_stats(tokens)
    # get most occured pair
    best_pair = max(stats,key=stats.get)
    rule_no = 256+i
    new_tokens = merge(tokens,best_pair,[],rule_no)
    merge_rules[best_pair] = rule_no
    new_pair = vocabulary[best_pair[0]]+vocabulary[best_pair[1]]
    vocabulary[rule_no] = new_pair
    tokens= new_tokens
    print("After new rule:",len(tokens))
    print("New rule ----->  ",(best_pair[0],best_pair[1]),":",new_pair)

    

Before new rule: 52
After new rule: 50
New rule ----->   (105, 115) : b'is'
Before new rule: 50
After new rule: 48
New rule ----->   (256, 32) : b'is '
Before new rule: 48
After new rule: 46
New rule ----->   (105, 114) : b'ir'
Before new rule: 46
After new rule: 44
New rule ----->   (115, 116) : b'st'
Before new rule: 44
After new rule: 42
New rule ----->   (114, 97) : b'ra'
Before new rule: 42
After new rule: 41
New rule ----->   (84, 104) : b'Th'
Before new rule: 41
After new rule: 40
New rule ----->   (261, 257) : b'This '
Before new rule: 40
After new rule: 39
New rule ----->   (262, 257) : b'This is '
Before new rule: 39
After new rule: 38
New rule ----->   (263, 116) : b'This is t'
Before new rule: 38
After new rule: 37
New rule ----->   (264, 104) : b'This is th'


In [49]:
tokens

[264,
 101,
 32,
 102,
 257,
 258,
 32,
 258,
 101,
 112,
 32,
 111,
 102,
 32,
 75,
 257,
 116,
 105,
 32,
 80,
 111,
 103,
 259,
 32,
 116,
 111,
 32,
 99,
 259,
 99,
 107,
 32,
 56,
 32,
 76,
 80,
 65]

In [54]:
bytes([65])

b'A'

In [55]:
vocabulary = {idx:bytes([idx]) for idx in range(256)}


In [65]:
def decode(ids):
    token_ids = [] 
    for idx in ids:
        token_ids.append(vocabulary[idx])
    raw_bytes = b"".join(token_ids)
    raw_bytes = raw_bytes.decode("utf-8",errors="replace")
    return raw_bytes


In [66]:
decode(tokens)

'This is the first step of Kirti Pogra to crack 8 LPA'

In [67]:
new_text = "This is Kirti"


In [85]:
def encode(text):
    ids = list(text.encode("utf-8"))
    while len(ids)>=2:
        stats = get_stats(ids)
        match_pairs= dict()
        for k,v in stats.items():
            pair_id = merge_rules.get(k)
            if pair_id:
                match_pairs[k] =  pair_id
        if not match_pairs:
            break
        best_pair = min(match_pairs,key=match_pairs.get)
        ids = merge(ids,best_pair,[],merge_rules[best_pair])

    return ids



In [86]:
token_ids = encode(new_text)
print(token_ids)
decode(token_ids)

[263, 75, 258, 116, 105]


'This is Kirti'

In [89]:
for i in token_ids:
    print(i,">>",vocabulary[i])

263 >> b'This is '
75 >> b'K'
258 >> b'ir'
116 >> b't'
105 >> b'i'


In [88]:
vocabulary[75]

b'K'

In [126]:
class BasicTokenizer:
    def __init__(self):
        self.merges = {}
        self.vocab = {idx: bytes([idx]) for idx in range(256)}
        pass

    def train(self,text,vocab_size,verbose=True):
        raw_bytes = text.encode('utf-8')
        tokens = list(raw_bytes)
        num_merges = vocab_size-256
        for i in range(num_merges):
            if verbose:
                print("Before new rule:", len(tokens))
            # calculate adjacent pairs 
            stats = self._get_stats(tokens)
            # get most occured pair
            best_pair = max(stats,key=stats.get)
            rule_no = 256+i
            new_tokens = self._merge(tokens,best_pair,rule_no)
            self.merges[best_pair] = rule_no
            new_pair = self.vocab[best_pair[0]]+self.vocab[best_pair[1]]
            self.vocab[rule_no] = new_pair
            tokens= new_tokens
            if verbose:
                print("After new rule:",len(tokens))
                print("New rule ----->  ",(best_pair[0],best_pair[1]),":",new_pair)

    def decode(self,ids):
        token_ids = [] 
        for idx in ids:
            token_ids.append(self.vocab[idx])
        raw_bytes = b"".join(token_ids)
        raw_bytes = raw_bytes.decode("utf-8",errors="replace")
        return raw_bytes
        

    def encode(self,text):
        ids = list(text.encode("utf-8"))
        while len(ids)>=2:
            stats = self._get_stats(ids)
            match_pairs= dict()
            for k in stats.keys():
                pair_id = self.merges.get(k)
                if pair_id:
                    match_pairs[k] =  pair_id
            if not match_pairs:
                break
            best_pair = min(match_pairs,key=match_pairs.get)
            ids = self._merge(ids,best_pair,self.merges[best_pair])
        
        return ids

    def save(self,file_name):
        model_file = f"{file_name}.model"

        with open(model_file,"w") as file:
            file.write("minbpe v1\n")
            for p0,p1 in self.merges:
                file.write(f"{p0} {p1}\n")

    def load(self,model_file):
        self.merges = {}
        self.vocab = {idx: bytes([idx]) for idx in range(256)}
        with open(model_file,"r") as file:
            header = file.readline()
            idx = 256
            for line in file:
                p0,p1 = map(int,line.strip().split())
                self.merges[(p0,p1)] = idx
                self.vocab[idx] = self.vocab[p0] + self.vocab[p1]
                idx +=1


    def _get_stats(self,tokens):
        total_len = len(tokens) -1
        adj_pairs = dict()
        for i in range(total_len):
            k = (tokens[i],tokens[i+1])
            v = adj_pairs.get(k,0) +1
            adj_pairs[k] =v

        return adj_pairs

    def _merge(self,ids, best_pair,new_idx):
        new_ids = []
        total_len = len(ids)
        i=0
        while i< total_len:
            if i<len(ids)-1 and (ids[i],ids[i+1]) == best_pair:
                new_ids.append(new_idx)  #255+1
                i+=2
            else:
                new_ids.append(ids[i])
                i+=1

        return new_ids

In [127]:
tokenizer = BasicTokenizer()

In [128]:
sample_text = """
Kirti Pogra is a passionate Artificial Intelligence and Generative AI engineer dedicated to mastering machine learning from first principles. 
By building Large Language Model components, Byte Pair Encoding tokenizers, and Transformer architectures from scratch in Python, 
Kirti is proving that deep technical understanding and hands-on coding lead to real-world innovation. 
Kirti Pogra is determined to crack high-impact engineering roles with 8+ LPA and beyond, transforming ideas into intelligent AI systems. 
This step-by-step journey of Kirti Pogra building AI models from scratch is the foundation for cracking 8+ LPA engineering roles.
"""

In [129]:
tokenizer.train(sample_text, vocab_size=300,verbose=False)


In [130]:

encoded = tokenizer.encode("This is Kirti")
print(len(encoded))
decoded = tokenizer.decode(encoded)
print("Decoded:", decoded)

5
Decoded: This is Kirti


In [131]:
tokenizer.save("mytokenizer")

In [132]:
# now check
tokenizer = BasicTokenizer()

In [133]:
# without loading
len(tokenizer.encode("This is Kirti"))

13

In [134]:
tokenizer.load("mytokenizer.model")

In [135]:
len(tokenizer.encode("This is Kirti"))

5

In [136]:
# 13 token to 5 token 